In [2]:
from __future__ import annotations
import dash  # L'import de la vraie librairie
from dash import dcc, html, Input, Output, State, ALL, ctx
import dash_bootstrap_components as dbc
import base64
from typing import Optional
import re
import io
import fitz  # PyMuPDF
import json
import uuid

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

def load_svg_as_data_uri(svg_path: str) -> Optional[str]:
    """Return a data URI for an SVG file, or None if not found."""
    try:
        with open(svg_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/svg+xml;base64,{b64}"
    except FileNotFoundError:
        return None

def extract_doi_from_pdf(text: str) -> Optional[str]:
    """Extract DOI from PDF text using regex pattern."""
    # Pattern for DOI: 10.xxxx/xxxxx
    doi_pattern = r'(?:https?://)?(?:www\.)?(?:dx\.)?doi\.org/|(?:doi:)\s*(?=10\.)|(10\.\S+/\S+)'
    match = re.search(r'(?:doi[:\s]+)?(?:https?://)?(?:dx\.)?doi\.org/(10\.\S+)', text, re.IGNORECASE)
    if match:
        return match.group(1) if match.group(1).startswith('10.') else match.group(0)

    # Alternative pattern
    match = re.search(r'10\.\d{4,}/\S+', text)
    if match:
        return match.group(0)
    return None


def extract_abstract_from_pdf(text: str) -> Optional[str]:
    """Extract abstract from PDF text."""
    # Look for "Abstract" section
    abstract_pattern = r'(?:abstract|summary)\s*[:]*\s*(.+?)(?=(?:introduction|keywords|1\.\s|methods|methodology|introduction|related work|background)|\Z)'
    match = re.search(abstract_pattern, text, re.IGNORECASE | re.DOTALL)
    if match:
        abstract_text = match.group(1).strip()
        # Clean up and limit to reasonable length
        abstract_text = re.sub(r'\s+', ' ', abstract_text)[:500]
        return abstract_text if len(abstract_text) > 20 else None
    return None


def extract_authors_from_pdf(text: str) -> list[dict]:
    """Extract authors by finding the typical author line in scientific papers."""
    authors = []

    def clean_name(name: str) -> str:
        # Supprime chiffres, *, †, § collés au nom
        return re.sub(r'[\d\*†‡§]+', '', name).strip()

    lines = text.split('\n')[:50]

    for line in lines:
        line = line.strip()

        # Une ligne d'auteurs contient typiquement "and" ou une virgule
        # et ressemble à des noms propres (Majuscule, pas trop longue)
        if len(line) > 150 or len(line) < 5:
            continue
        if not re.search(r'\band\b|,', line):
            continue
        # Doit commencer par une majuscule
        if not re.match(r'^[A-Z]', line):
            continue
        # Ne doit pas contenir de mots typiques de non-auteurs
        skip_words = ['abstract', 'keywords', 'introduction', 'figure',
                      'table', 'doi', 'http', 'university', 'institute',
                      'open access', 'copyright', 'license', 'received']
        if any(w in line.lower() for w in skip_words):
            continue
        # Tous les "mots" (après nettoyage) doivent ressembler à des noms propres
        # càd commencer par une majuscule ou être un chiffre/symbole
        test_line = clean_name(line)
        words = [w for w in re.split(r'[\s,]+', test_line) if w]
        if not words:
            continue
        # Au moins 80% des mots doivent commencer par une majuscule
        capitalized = sum(1 for w in words if re.match(r'^[A-Z]', w) or w.lower() == 'and')
        if capitalized / len(words) < 0.8:
            continue

        # C'est probablement une ligne d'auteurs — on parse
        raw_names = re.split(r',\s*|\s+and\s+', line)
        for raw in raw_names:
            name = clean_name(raw).strip()
            if not name or len(name) < 3:
                continue
            parts = name.split()
            if len(parts) >= 2:
                authors.append({
                    "name": " ".join(parts[:-1]),
                    "surname": parts[-1],
                    "email": ""
                })

    # Rattache l'email du corresponding author
    corr_match = re.search(
        r'\*Correspondence[:\s]+([A-Z][a-z]+(?:[\s\-][A-Za-z\-]+)+)\s+([\w.\-]+@[\w.\-]+\.\w+)',
        text
    )
    if corr_match and authors:
        corr_name = re.sub(r'[\d\*†‡§]+', '', corr_match.group(1)).strip()
        corr_email = corr_match.group(2)
        for author in authors:
            full = f"{author['name']} {author['surname']}"
            if corr_name in full or full in corr_name:
                author['email'] = corr_email

    return authors[:10]

def extract_date_from_pdf(text: str) -> Optional[str]:
    """Extract publication date from PDF text."""
    date_pattern = r'(\b\d{4}\b)'
    match = re.search(date_pattern, text)
    if match:
        return match.group(1)
    return None

def extract_text_by_blocks(uploaded_file_bytes) -> str:
    doc = fitz.open(stream=uploaded_file_bytes, filetype="pdf")
    full_text = ""
    for page in doc[:3]:
        # Trie les blocs par position verticale puis horizontale
        blocks = page.get_text("blocks")
        blocks.sort(key=lambda b: (round(b[1] / 20), b[0]))
        for block in blocks:
            full_text += block[4] + "\n"
    return full_text

def extract_pdf_metadata(uploaded_file) -> dict:
    """Extract metadata from PDF file."""
    metadata = {"doi": None, "abstract": None, "publication_date": None, "authors": []}
    try:
        # Extract text from PDF
        pdf_text = extract_text_by_blocks(uploaded_file.read())

        # Extract metadata
        metadata["doi"] = extract_doi_from_pdf(pdf_text)
        metadata["abstract"] = extract_abstract_from_pdf(pdf_text)
        metadata["authors"] = extract_authors_from_pdf(pdf_text)
        metadata["publication_date"] = extract_date_from_pdf(pdf_text)

    except Exception as e:
        print(f"Error processing PDF: {e}")
    return metadata


In [ ]:
def handle_pdf_upload(contents):
    if contents is None:
        return dash.no_update, dash.no_update, {}

    # Décodage et appel de votre fonction existante
    content_type, content_string = contents.split(',')
    decoded = base64.b64decode(content_string)

    # Appel de votre fonction : extract_pdf_metadata
    metadata = extract_pdf_metadata(io.BytesIO(decoded))

    return metadata.get('doi', ''), metadata.get('abstract', ''), metadata

handle_pdf_upload()